In [38]:
import pandas as pd
from sklearn.linear_model import Lasso
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

In [46]:
master_df = pd.read_csv('CocaColaSwire.csv')

C:\Users\Owner\AppData\Local\Temp\ipykernel_14804\262964533.py:1: DtypeWarning: Columns (8,11,12,13,14,15,16,17,18,20,21,22,23,24) have mixed types. Specify dtype option on import or set low_memory=False.
  master_df = pd.read_csv('CocaColaSwire.csv')


In [47]:
# GROUP 1: Matt, Alex, Carson
group_1_df = master_df.copy()

# GROUP 2: Nick S., Nick G.
group_2_df = master_df.copy()

In [50]:
#IMPORTANT
# REPLACE DF PLACEHOLDER
df_col_drop_lasso=group_2_df.drop(['ORDER_ID','PLANT_ID', 'FUNCTIONAL_AREA_NODE_1_MODIFIED', 
                                   'FUNCTIONAL_AREA_NODE_2_MODIFIED', 'FUNCTIONAL_AREA_NODE_3_MODIFIED', 'FUNCTIONAL_AREA_NODE_4_MODIFIED',
                                   'FUNCTIONAL_AREA_NODE_5_MODIFIED', 'EQUIP_VALID_TO', 'EXECUTION_START_DATE', 
                                   'EXECUTION_FINISH_DATE', 'EQUIP_START_UP_DATE', 'EQUIP_VALID_FROM', 'EQUIP_VALID_TO', 'ACTUAL_START_TIME', 'ACTUAL_FINISH_TIME'],axis=1)
#how to handle nodes? Equip start/vaid date?

df_filtered_lasso = df_col_drop_lasso.dropna(subset=['ORDER_DESCRIPTION', 'MAINTENANCE_PLAN', 'MAINTENANCE_ITEM','MAINTENANCE_TYPE_DESCRIPTION', 'FUNCTIONAL_LOC', 'EQUIPMENT_ID'], how='all')

df_filtered_lasso.isnull().sum()/len(df_filtered_lasso)


PRODUCTION_LOCATION             0.000000
ACTUAL_WORK_IN_MINUTES          0.000000
MAINTENANCE_PLAN                0.477051
MAINTENANCE_ITEM                0.477051
MAINTENANCE_ACTIVITY_TYPE       0.000000
ORDER_DESCRIPTION               0.000096
MAINTENANCE_TYPE_DESCRIPTION    0.000000
FUNCTIONAL_LOC                  0.000045
EQUIPMENT_ID                    0.000000
EQUIPMENT_DESC                  0.727730
EQUIP_CAT_DESC                  0.727730
dtype: float64

In [52]:
# Create a new DataFrame by dropping rows with NaN values
group_2_df_cleaned = df_filtered_lasso.dropna()

# Verify that there are no NaN values remaining
print("Number of NaN values in each column:\n", group_2_df_cleaned.isna().sum())

# Display the new shape of the DataFrame
print("Original shape:", group_2_df.shape)
print("Shape after dropping NaNs:", group_2_df_cleaned.shape)


Number of NaN values in each column:
 PRODUCTION_LOCATION             0
ACTUAL_WORK_IN_MINUTES          0
MAINTENANCE_PLAN                0
MAINTENANCE_ITEM                0
MAINTENANCE_ACTIVITY_TYPE       0
ORDER_DESCRIPTION               0
MAINTENANCE_TYPE_DESCRIPTION    0
FUNCTIONAL_LOC                  0
EQUIPMENT_ID                    0
EQUIPMENT_DESC                  0
EQUIP_CAT_DESC                  0
dtype: int64
Original shape: (1427264, 25)
Shape after dropping NaNs: (57067, 11)


In [54]:

# Separate features and target variable
X = group_2_df_cleaned.drop(columns=["ACTUAL_WORK_IN_MINUTES"])  # All columns except the target
y = group_2_df_cleaned["ACTUAL_WORK_IN_MINUTES"]  # Target variable

# Identify categorical and numerical columns
categorical_cols = X.select_dtypes(include=['object']).columns
numeric_cols = X.select_dtypes(exclude=['object']).columns

# Preprocessing for numeric data: Standard scaling
# Preprocessing for categorical data: One-hot encoding
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_cols)
    ]
)

# Create a pipeline with preprocessing and Lasso regression
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', Lasso(alpha=0.1))  # Adjust alpha as needed
])

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Fit the model
pipeline.fit(X_train, y_train)

# Make predictions and evaluate the model
y_pred = pipeline.predict(X_test)
mse = mean_squared_error(y_test, y_pred)

print("Mean Squared Error:", mse)
print("Lasso Coefficients:", pipeline.named_steps['regressor'].coef_)


Mean Squared Error: 5168.628753236943
Lasso Coefficients: [ 4.65339117  3.97683657  0.         ... -0.          0.
 -0.        ]


C:\Users\Owner\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:227: UserWarning: Found unknown categories in columns [1, 3, 5, 6] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [175]:
# Get the feature names after preprocessing
feature_names = numeric_cols.tolist() + pipeline.named_steps['preprocessor'].transformers_[1][1].get_feature_names_out(categorical_cols).tolist()

# Get the Lasso coefficients from the pipeline
lasso_coefficients = pipeline.named_steps['regressor'].coef_

# Combine feature names and coefficients into a DataFrame
coefficients_df = pd.DataFrame({'Feature': feature_names, 'Lasso Coefficient': lasso_coefficients})

# Filter out rows where the coefficient is zero
non_zero_coefficients_df = coefficients_df[coefficients_df['Lasso Coefficient'] != 0]

# Sort the DataFrame of non-zero coefficients in ascending order
sorted_coefficients_df = non_zero_coefficients_df.sort_values(by='Lasso Coefficient', ascending=True)

# Display the sorted non-zero coefficients 
print(sorted_coefficients_df)

                                                Feature  Lasso Coefficient
2230  MAINTENANCE_TYPE_DESCRIPTION_Preventive Mainte...        -241.419394
3                             PRODUCTION_LOCATION_MONZA         -21.694279
1100                        MAINTENANCE_PLAN_G816SC1447         -14.781044
11                        MAINTENANCE_PLAN_000000022943         -12.316223
302                          MAINTENANCE_PLAN_G29160018         -10.482835
...                                                 ...                ...
2258          FUNCTIONAL_LOC_G291-PRD-L15-L04-L120-MEAD          99.856683
1899  ORDER_DESCRIPTION_PASSIVATION OF L1 CONTACT TOWER         137.866944
1088                        MAINTENANCE_PLAN_G816SC1236         142.695444
306                          MAINTENANCE_PLAN_G29160028         171.811303
114                          MAINTENANCE_PLAN_G29110997         343.661445

[99 rows x 2 columns]


In [55]:
# Set display option to show all rows
pd.set_option('display.max_rows', None)

# Print the sorted DataFrame with all rows
print(sorted_coefficients_df)

# Reset display option to default (if desired)
pd.reset_option('display.max_rows')


NameError: name 'sorted_coefficients_df' is not defined

In [195]:
# Display the top 10 rows
print("Top 10 rows contributing to reduction of Actual Work in Minutes (smallest coefficients):")
print(sorted_coefficients_df.head(10))

# Display the bottom 10 rows
print("Top 10 rows contributing to increase of Actual Work in Minutes (largest coefficients):")
print(sorted_coefficients_df.tail(10))


Top 10 rows contributing to reduction of Actual Work in Minutes (smallest coefficients):
                                                Feature  Lasso Coefficient
2230  MAINTENANCE_TYPE_DESCRIPTION_Preventive Mainte...        -241.419394
3                             PRODUCTION_LOCATION_MONZA         -21.694279
1100                        MAINTENANCE_PLAN_G816SC1447         -14.781044
11                        MAINTENANCE_PLAN_000000022943         -12.316223
302                          MAINTENANCE_PLAN_G29160018         -10.482835
4                              PRODUCTION_LOCATION_ROMA          -9.770254
1604  ORDER_DESCRIPTION_KRONES LABELER #2 L3 FOR OPE...          -9.662708
2534       EQUIPMENT_DESC_L1 FILLER_ROTARY_BTL_60_VALVE          -9.596982
2311               FUNCTIONAL_LOC_G291-PRD-P05-XXX-S080          -9.547631
2254               FUNCTIONAL_LOC_G291-PRD-L15-L04-L070          -9.170004
Top 10 rows contributing to increase of Actual Work in Minutes (largest coefficients):